# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [37]:
import os
from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY was not found in Colab Secrets.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY loaded successfully into the environment.")


OPENAI_API_KEY loaded successfully into the environment.


In [38]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [39]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ['employees', 'salary', 'studies'],
    'definition': ['Employee information, name...', 'Salary details for each year', 'Educational studies, name of the institution, type of studies, level']
}
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [40]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [41]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [42]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [43]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables,
                                     question='How many employees are there in each department and what is their average salary?')

In [44]:
print(return_OAI(pqt1))

{
  "tables": ["employees", "salary"]
}


In [45]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question='What are the details of the employees who have more than 5 years of experience?')

In [46]:
print(return_OAI(pqt3))

{
    "tables": ["employees", "salary"]
}


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [51]:
questions = [
    "How many employees are there in each department and what is their average salary?",
    "What is the highest salary paid each year?",
    "Which employees have the highest salary within each education level?",
    "How many employees graduated from each university, and what is the average salary for each university?",
    "What is the average salary by education level?",
    "Which schools produced the highest paid employees?",
    "What is the total company revenue by department?",
    "What is the average education level of employees?"
]

for q in questions:
    prompt = prompt_question_tables.format(
        tables=text_tables,
        question=q
    )
    print("QUESTION:", q)
    print(return_OAI(prompt))
    print("-" * 80)

QUESTION: How many employees are there in each department and what is their average salary?
```json
{
    "tables": ["employees", "salary"]
}
```
--------------------------------------------------------------------------------
QUESTION: What is the highest salary paid each year?
```json
{
    "tables": ["employees", "salary"]
}
```
--------------------------------------------------------------------------------
QUESTION: Which employees have the highest salary within each education level?
```json
{
  "tables": ["employees", "salary", "studies"]
}
```
--------------------------------------------------------------------------------
QUESTION: How many employees graduated from each university, and what is the average salary for each university?
```json
{
    "tables": ["employees", "studies", "salary"]
}
```
--------------------------------------------------------------------------------
QUESTION: What is the average salary by education level?
```json
{
  "tables": ["employees", "salary", 

In [52]:
prompt_question_tables_v2 = """
You are a data analyst helping select database tables for a SQL query.

Given these table names and definitions:

### Tables
{tables}

Task:
Identify only the tables needed to answer the user question.

Rules:
- Do not include tables that are not listed.
- Do not invent table names.
- If the question cannot be answered with the available tables, return an empty list.
- Use this format:
{{
  "tables": [],
  "reason": ""
}}

### User question
{question}
"""

In [53]:
for q in questions:
    prompt = prompt_question_tables_v2.format(
        tables=text_tables,
        question=q
    )
    print("QUESTION:", q)
    print(return_OAI(prompt))
    print("-" * 80)

QUESTION: How many employees are there in each department and what is their average salary?
{
  "tables": ["employees", "salary"],
  "reason": "The 'employees' table contains information about employees, including their department. The 'salary' table contains salary details which can be used to calculate the average salary per department."
}
--------------------------------------------------------------------------------
QUESTION: What is the highest salary paid each year?
{
  "tables": ["salary"],
  "reason": "The salary table contains the salary details for each year, which is needed to determine the highest salary paid each year."
}
--------------------------------------------------------------------------------
QUESTION: Which employees have the highest salary within each education level?
{
  "tables": ["employees", "salary", "studies"],
  "reason": "The tables 'employees', 'salary', and 'studies' are needed to retrieve the employee information, salary details, and educational stud

## Conclusions

Both Prompt 1 and Prompt 2 returned the same tables for all questions. This shows that the model was able to understand the table descriptions and consistently choose the relevant tables.

The improved prompt did not change the results, but it provided clearer instructions and more structured output.

One limitation was that the model selected tables for a revenue-related question even though no revenue table was available. This shows that LLMs can sometimes make assumptions when the database schema is incomplete.

Overall, the exercise showed that LLMs can be useful for identifying relevant tables before generating SQL queries, especially when clear table descriptions are provided.
